# Fase 4A — Agregasi `demand_zone_hour` (NYC Yellow Taxi 2026)

**Penanggung jawab:** Mufid Dhamarjati Kusuma · **Periode:** Januari–Juni 2026 · **Sumber:** `main_trips` hasil cleaning Anggota 1.

Notebook ini **khusus fase 4A**, tidak mengulang EDA dan tidak mengubah data mentah. Hasilnya digunakan oleh Fase 4B, Fase 5, dan modeling.

**Ketentuan wajib**
- Unit analisis: **centroid zona pickup** (`PU_longitude`, `PU_latitude`) × 1 jam lokal New York.
- Grid lengkap: kombinasi zona-jam valid tanpa pickup = **0**.
- `2026-03-08 02:00` (DST *spring forward*) = **tidak tersedia**, `pickup_count = NaN`, `is_dst_missing = 1`.
- Total agregasi dari seluruh jam valid **harus sama** dengan baris perjalanan valid pada `main_trips`.
- Metadata zona dari `datasets/zone/zone_centroid_lookup.csv`.

> Data yang berasal dari `main_trips` diasumsikan sudah lolos cleaning. Notebook **tidak** membaca Parquet mentah (`datasets/raw`) agar tidak menggandakan aturan cleaning anggota lain. Perhitungan dibagi menjadi batch agar lebih hemat memori.

## 0. Import dan konfigurasi (path mengikuti folder `script/`)

In [1]:
from pathlib import Path
import gc
import time
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# Cari root proyek secara aman saat notebook dibuka dari script/ atau root repository.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "datasets").is_dir() and (p / "script").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Root proyek tidak ditemukan. Buka notebook dari folder repository (atau script/).")

INTERIM = ROOT / "datasets" / "interim" / "2026"
LOOKUP_CSV = ROOT / "datasets" / "zone" / "zone_centroid_lookup.csv"
OUTPUT_PATH = INTERIM / "demand_zone_hour.parquet"
QA_DIR = ROOT / "out" / "output_eda"

START = pd.Timestamp("2026-01-01 00:00:00")
END_EXCLUSIVE = pd.Timestamp("2026-07-01 00:00:00")
DST_MISSING = pd.Timestamp("2026-03-08 02:00:00")
BATCH_SIZE = 250_000
ROUND_DECIMALS = 6   # Selaras dengan notebook 05_features.ipynb saat ini

INTERIM.mkdir(parents=True, exist_ok=True)
QA_DIR.mkdir(parents=True, exist_ok=True)
print("Root proyek :", ROOT)
print("Input       :", INTERIM)
print("Tabel zona  :", LOOKUP_CSV)
print("Output      :", OUTPUT_PATH)

Root proyek : C:\Users\hp\Documents\25_IF-C_Cooked_pake_d
Input       : C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets\interim\2026
Tabel zona  : C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets\zone\zone_centroid_lookup.csv
Output      : C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets\interim\2026\demand_zone_hour.parquet


## 1. Temukan `main_trips` hasil cleaning

Mendukung `main_trips.parquet`, folder `main_trips/` berisi Parquet, atau sekumpulan `main_trips_*.parquet`. Tidak otomatis beralih ke file mentah apabila data cleaning belum tersedia.

In [2]:
candidates = [INTERIM / "main_trips.parquet", INTERIM / "main_trips"]
input_paths = []
for p in candidates:
    if p.is_file() or (p.is_dir() and any(p.rglob("*.parquet"))):
        input_paths = [p]
        break
if not input_paths:
    input_paths = sorted(INTERIM.glob("main_trips_*.parquet"))

if not input_paths:
    raise FileNotFoundError(
        f"main_trips hasil cleaning belum ditemukan di {INTERIM}. "
        "Minta Anggota 1 menyiapkan main_trips.parquet atau file main_trips_*.parquet."
    )
print("Sumber main_trips:", *input_paths, sep="\n - ")

source = ds.dataset(str(input_paths[0]) if len(input_paths) == 1 else [str(p) for p in input_paths], format="parquet")
NEEDED = ["tpep_pickup_datetime", "PU_longitude", "PU_latitude"]
missing = sorted(set(NEEDED) - set(source.schema.names))
if missing:
    raise KeyError(f"Kolom wajib belum ada pada main_trips: {missing}. Kolom yang ada: {source.schema.names}")
print("Skema sumber OK.")

Sumber main_trips:
 - C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets\interim\2026\main_trips.parquet
Skema sumber OK.


## 2. Load dan verifikasi lookup centroid zona

In [3]:
if not LOOKUP_CSV.exists():
    raise FileNotFoundError(f"Tabel centroid tidak ditemukan: {LOOKUP_CSV}")

zona = pd.read_csv(LOOKUP_CSV)
required_zone = ["LocationID", "longitude", "latitude", "zone_name", "borough", "is_airport", "is_cbd"]
missing_zone = sorted(set(required_zone) - set(zona.columns))
if missing_zone:
    raise KeyError(f"Kolom lookup tidak lengkap: {missing_zone}")

zona = zona[required_zone].copy()
zona = zona.dropna(subset=["longitude", "latitude"])
zona = zona.loc[zona["LocationID"].between(1, 263)].copy()
zona = zona.rename(columns={
    "LocationID": "zone_id", "longitude": "PU_longitude", "latitude": "PU_latitude",
    "zone_name": "PU_zone_name", "borough": "PU_borough"
})
zona["lo6"] = zona["PU_longitude"].round(ROUND_DECIMALS)
zona["la6"] = zona["PU_latitude"].round(ROUND_DECIMALS)
assert zona["zone_id"].is_unique, "Tabel centroid memiliki zone_id duplikat"
assert not zona[["lo6", "la6"]].duplicated().any(), "Koordinat centroid tidak unik setelah pembulatan 6 digit"
assert zona[["PU_longitude", "PU_latitude"]].notna().all().all(), "Terdapat koordinat zona kosong"
print(f"Zona valid: {len(zona):,} (IDs Unknown/Outside dikecualikan)")
display(zona.head())

Zona valid: 263 (IDs Unknown/Outside dikecualikan)


,zone_id,PU_longitude,PU_latitude,PU_zone_name,PU_borough,is_airport,is_cbd,lo6,la6
0,1,-74.174000,40.691831,Newark Airport,EWR,1,0,-74.174000,40.691831
1,2,-73.831299,40.616745,Jamaica Bay,Queens,0,0,-73.831299,40.616745
2,3,-73.847422,40.864474,Allerton/Pelham Gardens,Bronx,0,0,-73.847422,40.864474
3,4,-73.976968,40.723752,Alphabet City,Manhattan,0,1,-73.976968,40.723752
4,5,-74.188484,40.552659,Arden Heights,Staten Island,0,0,-74.188484,40.552659


## 3. Agregasi pickup per zona-jam dengan batch

Waktu pada TLC Parquet diasumsikan sebagai **waktu dinding lokal New York** (timestamp tanpa offset). Jangan melakukan `tz_localize` sembarangan; timestamp `2026-03-08 02:00–02:59` adalah jam tidak valid dan akan ditolak agar ketidakcocokan data tidak tersembunyi.

Semua data masuk harus memiliki pickup dalam Januari–Juni 2026 dan cocok dengan zona centroid resmi. Kalau tidak, notebook menghentikan proses dan meminta pemeriksaan sumber cleaning.

In [4]:
t0 = time.perf_counter()
chunks = []
rows_source = 0
bad_time = bad_zone = bad_dst = 0
sample_bad_zone = None
zone_keys = pd.MultiIndex.from_frame(zona[["lo6", "la6"]])

scanner = source.scanner(columns=NEEDED, batch_size=BATCH_SIZE, use_threads=True)
for batch_no, batch in enumerate(scanner.to_batches(), start=1):
    batch_df = batch.to_pandas()
    n = len(batch_df)
    rows_source += n
    raw_hour = pd.to_datetime(batch_df["tpep_pickup_datetime"], errors="coerce")
    if getattr(raw_hour.dt, "tz", None) is not None:
        raw_hour = raw_hour.dt.tz_convert("America/New_York").dt.tz_localize(None)

    outside = raw_hour.isna() | (raw_hour < START) | (raw_hour >= END_EXCLUSIVE)
    bad_time += int(outside.sum())
    if outside.any():
        raise ValueError(f"Batch {batch_no}: {outside.sum()} pickup tidak valid/di luar Jan–Jun 2026. Periksa main_trips.")

    hours = raw_hour.dt.floor("h")
    nonexistent = hours.eq(DST_MISSING)
    bad_dst += int(nonexistent.sum())
    if nonexistent.any():
        raise ValueError(f"Batch {batch_no}: ditemukan {nonexistent.sum()} pickup pada jam DST yang tidak ada ({DST_MISSING}). Koreksi sumber waktu sebelum agregasi.")

    lo = pd.to_numeric(batch_df["PU_longitude"], errors="coerce").round(ROUND_DECIMALS)
    la = pd.to_numeric(batch_df["PU_latitude"], errors="coerce").round(ROUND_DECIMALS)
    observed_keys = pd.MultiIndex.from_arrays([lo, la])
    unknown = ~observed_keys.isin(zone_keys)
    bad_zone += int(unknown.sum())
    if unknown.any():
        sample_bad_zone = batch_df.loc[unknown, ["PU_longitude", "PU_latitude"]].head(5)
        raise ValueError(f"Batch {batch_no}: {unknown.sum()} baris tidak cocok dengan lookup centroid. Contoh:\n{sample_bad_zone}")

    grouped = (pd.DataFrame({"lo6": lo.to_numpy(), "la6": la.to_numpy(), "hour": hours.to_numpy()})
               .groupby(["lo6", "la6", "hour"], sort=False, observed=True)
               .size().rename("pickup_count").reset_index())
    chunks.append(grouped)
    del batch_df, grouped
    if batch_no % 10 == 0:
        print(f"Batch {batch_no:>3}: {rows_source:,} perjalanan telah diproses")

del scanner
if rows_source == 0:
    raise ValueError("main_trips kosong")

counts = (pd.concat(chunks, ignore_index=True)
          .groupby(["lo6", "la6", "hour"], as_index=False, sort=False, observed=True)["pickup_count"].sum())
del chunks
gc.collect()
assert int(counts["pickup_count"].sum()) == rows_source, "Agregasi sementara tidak sama dengan sumber"
print(f"Baris perjalanan sumber : {rows_source:,}")
print(f"Kombinasi zona-jam aktif: {len(counts):,}")
print(f"Waktu agregasi          : {time.perf_counter()-t0:.1f} detik")

Batch  10: 2,097,152 perjalanan telah diproses
Batch  20: 4,194,304 perjalanan telah diproses
Batch  30: 6,291,456 perjalanan telah diproses
Batch  40: 8,388,608 perjalanan telah diproses
Batch  50: 10,485,760 perjalanan telah diproses
Batch  60: 12,582,912 perjalanan telah diproses
Batch  70: 14,680,064 perjalanan telah diproses
Batch  80: 16,777,216 perjalanan telah diproses
Batch  90: 18,874,368 perjalanan telah diproses
Batch 100: 20,971,520 perjalanan telah diproses
Baris perjalanan sumber : 22,274,550
Kombinasi zona-jam aktif: 685,384
Waktu agregasi          : 10.3 detik


## 4. Bangun grid zona × jam lengkap

Ada 4.344 jam kalender naif dalam rentang Januari–Juni 2026. Jam `2026-03-08 02:00` di New York **tidak terjadi**, sehingga diberi label khusus dan tidak diisi nol. Bagi rentang Jan–Jun tidak ada pengulangan jam DST musim gugur.

In [5]:
hours = pd.date_range(START, END_EXCLUSIVE, freq="h", inclusive="left")
assert DST_MISSING in hours
assert len(hours) == 4344, "Rentang waktu berbeda dari Jan–Jun 2026"

# Cross-join semua zona ke setiap jam, lalu left-join hasil pickup aktual.
grid = zona.merge(pd.DataFrame({"hour": hours}), how="cross")
grid = grid.merge(counts, on=["lo6", "la6", "hour"], how="left", validate="one_to_one", indicator=True)

unmapped = counts.merge(zona[["lo6", "la6"]], on=["lo6", "la6"], how="left", indicator=True)
assert not unmapped["_merge"].eq("left_only").any(), "Ada agregat dengan zona tidak dikenali"
assert int(grid.loc[grid["_merge"].eq("both"), "pickup_count"].sum()) == rows_source, "Total pickup hilang saat merge ke grid"

grid["is_dst_missing"] = grid["hour"].eq(DST_MISSING).astype("int8")
grid["pickup_count"] = grid["pickup_count"].fillna(0).astype("float64")
grid.loc[grid["is_dst_missing"].eq(1), "pickup_count"] = np.nan

# Kolom yang digunakan oleh feature engineering anggota 1.
OUTPUT_COLUMNS = ["hour", "zone_id", "PU_longitude", "PU_latitude", "PU_zone_name",
                  "PU_borough", "is_airport", "is_cbd", "pickup_count", "is_dst_missing"]
result = grid[OUTPUT_COLUMNS].sort_values(["zone_id", "hour"]).reset_index(drop=True)
print(f"Jam kalender termasuk DST placeholder: {len(hours):,}")
print(f"Baris grid                         : {len(result):,}")
print(f"Baris placeholder DST              : {int(result['is_dst_missing'].sum()):,}")
print(f"Baris nol valid                    : {int(((result['pickup_count']==0)&(result['is_dst_missing']==0)).sum()):,}")

Jam kalender termasuk DST placeholder: 4,344
Baris grid                         : 1,142,472
Baris placeholder DST              : 263
Baris nol valid                    : 456,825


## 5. Quality gate — seluruh pengujian harus lolos

In [7]:
n_zones = len(zona)
assert len(result) == n_zones * len(hours), "Grid zona-jam tidak lengkap"
assert not result.duplicated(["zone_id", "hour"]).any(), "Duplikasi key zona-jam"
assert result.groupby("zone_id")["hour"].nunique().eq(len(hours)).all(), "Ada zona dengan jam tidak lengkap"
assert result["is_dst_missing"].sum() == n_zones, "Placeholder DST tidak berjumlah 1 per zona"
assert result.loc[result["is_dst_missing"].eq(1), "pickup_count"].isna().all(), "DST harus NaN"
assert result.loc[result["is_dst_missing"].eq(0), "pickup_count"].notna().all(), "Jam valid tidak boleh NaN"
assert (result["pickup_count"].dropna() >= 0).all(), "Ada pickup negatif"
assert np.allclose(result["pickup_count"].dropna() % 1, 0), "Pickup count harus bilangan cacah"

agg_total = int(result["pickup_count"].sum())
assert agg_total == rows_source, f"Total pickup agregat {agg_total:,} != sumber {rows_source:,}"

# Audit per bulan: output agregasi harus identik dengan ringkasan dari counts.
monthly_source = counts.assign(month=counts["hour"].dt.to_period("M").astype(str)).groupby("month")["pickup_count"].sum()
monthly_output = result.assign(month=result["hour"].dt.to_period("M").astype(str)).groupby("month")["pickup_count"].sum()
monthly = pd.DataFrame({"main_trips": monthly_source, "demand_zone_hour": monthly_output}).fillna(0)
monthly["difference"] = monthly["demand_zone_hour"] - monthly["main_trips"]
assert monthly["difference"].eq(0).all(), "Jumlah pickup per bulan berbeda"
assert len(monthly) == 6, "Data sumber tidak mencakup keenam bulan (Jan–Jun)"

print("SEMUA QUALITY GATE LOLOS ")
print(f"Sumber : {rows_source:,} pickup | Agregat: {agg_total:,} pickup | Selisih: {agg_total-rows_source}")
display(monthly)

SEMUA QUALITY GATE LOLOS 
Sumber : 22,274,550 pickup | Agregat: 22,274,550 pickup | Selisih: 0


,main_trips,demand_zone_hour,difference
month,,,
2026-01,3615518,3615518.0,0.0
2026-02,3310826,3310826.0,0.0
2026-03,3858721,3858721.0,0.0
2026-04,3743693,3743693.0,0.0
2026-05,3999128,3999128.0,0.0
2026-06,3746664,3746664.0,0.0


## 6. Simpan hasil dan laporan audit

In [8]:
# Jangan menimpa hasil resmi apabila quality gate belum lolos (cell sebelumnya wajib dijalankan).
result.to_parquet(OUTPUT_PATH, index=False)
monthly.to_csv(QA_DIR / "04a_aggregation_monthly_check.csv", index_label="month")

summary = pd.DataFrame([{
    "source": "; ".join(str(x) for x in input_paths),
    "total_pickup_source": rows_source,
    "total_pickup_aggregation": agg_total,
    "difference": agg_total - rows_source,
    "n_zones": n_zones,
    "n_hours_including_dst_placeholder": len(hours),
    "n_dst_placeholder_rows": int(result["is_dst_missing"].sum()),
    "n_zero_valid_rows": int(((result["pickup_count"]==0) & result["is_dst_missing"].eq(0)).sum()),
    "n_grid_rows": len(result),
    "coordinate_key_round_decimals": ROUND_DECIMALS,
    "timezone_convention": "America/New_York local wall time (naive)",
    "dst_nonexistent_hour": str(DST_MISSING),
}])
summary.to_csv(QA_DIR / "04a_aggregation_quality_report.csv", index=False)

# Uji read-back sebagai verifikasi file yang tersimpan
saved = pd.read_parquet(OUTPUT_PATH)
assert len(saved) == len(result)
assert int(saved["pickup_count"].sum()) == rows_source
print("TERSIMPAN:")
print("-", OUTPUT_PATH)
print("-", QA_DIR / "04a_aggregation_monthly_check.csv")
print("-", QA_DIR / "04a_aggregation_quality_report.csv")
print("\nSilakan serahkan output demand_zone_hour.parquet ke Anggota 1 dan Anggota 3.")

TERSIMPAN:
- C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\datasets\interim\2026\demand_zone_hour.parquet
- C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\out\output_eda\04a_aggregation_monthly_check.csv
- C:\Users\hp\Documents\25_IF-C_Cooked_pake_d\out\output_eda\04a_aggregation_quality_report.csv

Silakan serahkan output demand_zone_hour.parquet ke Anggota 1 dan Anggota 3.


## 7. Catatan handoff untuk anggota tim

**Format output** `datasets/interim/2026/demand_zone_hour.parquet`:

| Kolom | Keterangan |
|---|---|
| `hour` | Awal jam, waktu lokal New York tanpa offset |
| `zone_id` | LocationID dari lookup, digunakan untuk pemetaan |
| `PU_longitude`, `PU_latitude` | Koordinat centroid pickup, bukan GPS penumpang |
| `PU_zone_name`, `PU_borough` | Metadata zona |
| `is_airport`, `is_cbd` | Atribut zona |
| `pickup_count` | Jumlah pickup pada zona-jam; 0 bila valid tanpa pickup, NaN saat DST hilang |
| `is_dst_missing` | 1 hanya pada `2026-03-08 02:00`, selainnya 0 |

**Koordinasi yang diperlukan:** saat ini `script/05_features.ipynb` masih membangun dan menulis ulang `demand_zone_hour.parquet`. Minta Anggota 1 menyesuaikan Fase 5 untuk **membaca hasil Fase 4A ini**, bukan menimpa file. Jangan menyatukan `main_trips` mentah dan hasil cleaning dalam satu agregasi.

**Keterbatasan:** jika pada masa depan periode data melampaui November 2026, jam lokal berulang saat DST berakhir harus dipisahkan dengan offset UTC / timezone-aware; desain naive dengan satu placeholder ini khusus Jan–Jun 2026.